In [ ]:
# https://medium.com/@nomannayeem/comprehensive-guide-to-time-series-data-analytics-and-forecasting-with-python-2c82de2c8517


#### **Time Series Forecasting with MFLES (StatsForecast)**



 What is MFLES?

* **MFLES = Median + Fourier + Linear + Exponential Smoothing**
* A **hybrid method** in *StatsForecast* by Nixtla.
* Built on **Gradient Boosted Time Series Decomposition**.
* Recently top-performing in **VN1 Forecasting Competition** and competitive in **M4 Benchmark**.



 Why MFLES? (Compared to ARIMA/ETS/Theta)

* Traditional ARIMA/ETS handle trend & seasonality but struggle with:

  * Multiple seasonalities
  * Trend changepoints
  * Exogenous features
* **MFLES** extends decomposition by applying **boosting**:

  * Residuals passed across iterations (like gradient boosting).
  * Component-wise learning rates (trend, seasonality, exogenous).
  * Can fit **infinite seasonalities** & **multiple changepoints**.



 Core Idea: Gradient Boosted Decomposition

1. Start with decomposition: Trend + Seasonality + Residual.
2. Treat each component as a **base learner** in a boosting framework.
3. Apply **learning rates per component** (avoid overfitting).
4. Iterate to refine → better generalization.

👉 Enables:

* Detecting seasonalities round by round.
* Capturing multiple changepoints.
* Plugging in "exotic" estimators (Fourier, Median, Linear, ETS).



 Implementation in Python


 Key Parameters

* **season\_length** → list of seasonal periods (e.g., `[12]` for monthly).
* **test\_size** → periods in CV folds; usually main season length.
* **n\_windows** → # of CV windows (controls robustness).
* **metric** → optimization target (`smape`, `rmse`, `mae`, `mape`).


 Benchmark (M4 Competition)

* Dataset: 100,000 series (yearly, quarterly, monthly, weekly, daily, hourly).
* Domains: Demographics, Finance, Industry, Macro, Micro.
* **Results**:

  * MFLES ranked **2nd overall**, just behind Theta.
  * Best method in **2 out of 6 frequencies**.
  * More flexible than Theta (handles multiple seasonality, changepoints, exogenous).


 Strengths of MFLES

✅ Handles multiple seasonalities.
✅ Flexible with changepoints.
✅ Supports exogenous features.
✅ Outperforms ARIMA/ETS in benchmarks.



 Limitations

⚠️ More parameters to tune than "pure auto" models.
⚠️ Risk of overfitting without careful test\_size & learning rates.
⚠️ Still relatively new → less field adoption than ARIMA/ETS.


Future Directions

* **Exogenous any-estimator support** → Use ML models (XGBoost, SVMs, Neural Nets) to model exogenous components inside MFLES.


MFLES = **Boosted decomposition method** combining Median, Fourier, Linear, and ETS.
It generalizes seasonal-trend decomposition → making it flexible, competitive, and **state-of-the-art in statistical forecasting**.
Best used when:

* Multiple seasonalities
* Data with trend breaks
* Forecasts with external regressors


In [ ]:
! uv pip install pygments -U --force-reinstall
! uv pip install statsforecast 

: 

In [ ]:


import pandas as pd
import numpy as np
from statsforecast.models import AutoMFLES
import matplotlib.pyplot as plt

# Load Airline Passengers dataset
df = pd.read_csv('https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv')
y = df['Passengers'].values

# Fit AutoMFLES
mfles_model = AutoMFLES(
    season_length=[12],   # important seasonalities
    test_size=12,         # CV test fold size
    n_windows=2,          # # of CV windows
    metric='smape'        # optimization metric
)
mfles_model.fit(y=y)

# Forecast
predicted = mfles_model.predict(12)['mean']
fitted = mfles_model.predict_in_sample()['fitted']

plt.plot(np.append(fitted, predicted), linestyle='dashed', color='red')
plt.plot(y)
plt.show()